# Deteksi dan Estimasi Jumlah Unit Keramba Jaring Apung (KJA)
## dari Citra Satelit Top-View untuk Pemantauan Daya Dukung Waduk Jatiluhur

Tugas Besar Pengolahan Citra Digital · Kelompok: Akhsya, Dayan, Aufa

**Alur metode (tanpa deep learning):**
1. Preprocessing: grayscale → median filter → CLAHE
2. Ekstraksi fitur: tekstur GLCM (contrast, homogeneity, energy, entropy), warna hijau relatif, dan kerapatan garis rangka (white top-hat)
3. Segmentasi K-Means (k = 3)
4. Post-processing: syarat kerapatan garis, pemangkasan tepi ke rangka, morfologi, filter luas
5. Luas KJA dan estimasi jumlah unit
6. Evaluasi terhadap kunci jawaban (IoU, precision, recall, F1)

**Cara pakai:** jalankan sel dari atas ke bawah (`Runtime → Run all`). Di bagian **Upload**, pilih citra potongan dan kunci jawabannya.

## 0. Kode fungsi
Sel ini menyimpan semua fungsi ke file `kja_pipeline.py`. Cukup dijalankan, tidak perlu diubah.

In [ ]:
%%writefile kja_pipeline.py
"""
kja_pipeline.py — Deteksi dan estimasi jumlah unit Keramba Jaring Apung (KJA)
dari citra satelit top-view dengan metode klasik (tanpa deep learning).

Alur: preprocessing -> fitur tekstur GLCM + fitur hijau relatif + kerapatan garis rangka
      -> K-Means -> post-processing morfologi -> luas & estimasi jumlah unit -> evaluasi.
"""
import time
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

GLCM_PROPS = ["contrast", "homogeneity", "energy", "entropy"]


# ------------------------------------------------------------------ 1. citra & preprocessing
def load_image(path, crop_bottom=0):
    """Baca citra BGR. crop_bottom = jumlah baris bawah yang dibuang (logo Google Earth / garis skala)."""
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(path)
    return bgr[:-crop_bottom] if crop_bottom else bgr


def preprocess(bgr, median_ksize=3, clahe_clip=2.0, clahe_tile=8):
    """Grayscale -> median filter (redam noise) -> CLAHE (naikkan kontras lokal)."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    den = cv2.medianBlur(gray, median_ksize)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(clahe_tile, clahe_tile))
    return clahe.apply(den)


# ------------------------------------------------------------------ 2. fitur
def glcm_maps(enh, win=15, step=4, levels=32, distance=1, verbose=True):
    """Peta fitur GLCM dengan jendela geser. Dihitung tiap `step` piksel lalu diperbesar ke ukuran citra."""
    h, w = enh.shape
    q = (enh.astype(np.float32) / 256 * levels).astype(np.uint8)
    pad = win // 2
    qp = cv2.copyMakeBorder(q, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    ys, xs = np.arange(0, h, step), np.arange(0, w, step)
    out = {p: np.zeros((len(ys), len(xs)), np.float32) for p in GLCM_PROPS}
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
    t = time.time()
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            G = graycomatrix(qp[y:y + win, x:x + win], [distance], angles,
                             levels=levels, symmetric=True, normed=True)
            for p in ("contrast", "homogeneity", "energy"):
                out[p][i, j] = graycoprops(G, p).mean()
            P = G.mean(axis=3)[:, :, 0]
            P = P[P > 0]
            out["entropy"][i, j] = -(P * np.log2(P)).sum()
        if verbose and (i % max(1, len(ys) // 10) == 0):
            print(f"  GLCM baris {i + 1}/{len(ys)} ({time.time() - t:.0f} detik)")
    if verbose:
        print(f"  GLCM selesai: {len(ys) * len(xs)} jendela, {time.time() - t:.0f} detik")
    return {k: cv2.resize(v, (w, h), interpolation=cv2.INTER_LINEAR) for k, v in out.items()}


def green_relative(bgr, win=15):
    """Excess Green (2G-R-B) rata-rata lokal, dikurangi latar sekitarnya.
    Dikurangi latar supaya beda warna antar-citra satelit (mosaik) tidak dianggap tanaman."""
    h, w = bgr.shape[:2]
    f = bgr.astype(np.float32) / 255
    exg = 2 * f[:, :, 1] - f[:, :, 2] - f[:, :, 0]
    local = cv2.blur(exg, (win, win))
    small = cv2.resize(exg, (max(1, w // 8), max(1, h // 8)), interpolation=cv2.INTER_AREA)
    u8 = (np.clip(small + 1, 0, 2) * 127).astype(np.uint8)
    bg = cv2.resize(cv2.medianBlur(u8, 31), (w, h)).astype(np.float32) / 127 - 1
    return local - bg


def line_density(bgr, enh, win=15, tophat_size=5, max_green=0.05):
    """Kerapatan garis rangka: struktur terang yang lebih tipis dari `tophat_size` piksel
    (white top-hat + ambang Otsu) dan tidak hijau, dirata-ratakan dalam jendela."""
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (tophat_size, tophat_size))
    th = cv2.morphologyEx(enh, cv2.MORPH_TOPHAT, k)
    t, _ = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    f = bgr.astype(np.float32) / 255
    exg = 2 * f[:, :, 1] - f[:, :, 2] - f[:, :, 0]
    lines = ((th > t) & (exg < max_green)).astype(np.float32)
    return cv2.blur(lines, (win, win))


# ------------------------------------------------------------------ 3. segmentasi
def _smooth_labels(lbl, k, size=15):
    votes = np.stack([cv2.blur((lbl == c).astype(np.float32), (size, size)) for c in range(k)], -1)
    return votes.argmax(-1)


def segment_kmeans(features, k=3, kja_by="line_density", sample=200000, seed=0):
    """K-Means pada tumpukan fitur (dict nama -> peta 2D).
    Klaster KJA = klaster dengan rata-rata fitur `kja_by` tertinggi."""
    names = list(features)
    h, w = features[names[0]].shape
    X = np.stack([features[n] for n in names], -1).reshape(-1, len(names))
    Xs = StandardScaler().fit_transform(X)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Xs), min(sample, len(Xs)), replace=False)
    km = KMeans(n_clusters=k, n_init=5, random_state=seed).fit(Xs[idx])
    labels = _smooth_labels(km.predict(Xs).reshape(h, w), k)
    ref = features[kja_by]
    means = [ref[labels == c].mean() if (labels == c).any() else -np.inf for c in range(k)]
    kja_cluster = int(np.argmax(means))
    summary = []
    for c in range(k):
        m = labels == c
        summary.append({"klaster": c, "persen": round(m.mean() * 100, 1),
                        **{n: round(float(features[n][m].mean()), 3) for n in names}})
    return labels, kja_cluster, summary


def select_kja_clusters(summary, feature="line_density", frac=0.4):
    """Klaster dianggap KJA bila rata-rata kerapatan garisnya >= frac x klaster tertinggi.
    Perlu karena baris keramba yang garisnya samar kadang dipisah K-Means ke klaster sendiri."""
    vals = [s[feature] for s in summary]
    top = max(vals)
    return [s["klaster"] for s in summary if s[feature] >= frac * top]


def line_gate(line_dens, win=31, threshold=0.06):
    """Syarat tambahan: kerapatan garis dalam jendela lebar (±27 m) harus > threshold.
    Membuang hamparan tanaman yang bertekstur tetapi tidak punya garis rangka."""
    return cv2.blur(line_dens, (win, win)) > threshold


def raw_lines(enh, tophat_size=5):
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (tophat_size, tophat_size))
    th = cv2.morphologyEx(enh, cv2.MORPH_TOPHAT, k)
    t, _ = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th > t


def refine_to_frame(mask, enh, close_px=13):
    """Pangkas tepi: garis rangka di dalam mask ditutup (closing) menjadi bidang keramba,
    lalu diiriskan dengan mask. Hasilnya batas KJA menempel ke rangka, tidak melebar ke air."""
    lines = (raw_lines(enh) & mask).astype(np.uint8)
    hull = cv2.morphologyEx(lines, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (close_px, close_px)))
    hull = cv2.morphologyEx(hull, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    return hull.astype(bool) & mask


def color_baseline(bgr):
    """Pembanding: K-Means k=2 hanya dengan warna (Lab). Klaster lebih terang = 'KJA'."""
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    rng = np.random.default_rng(0)
    idx = rng.choice(len(lab), min(200000, len(lab)), replace=False)
    km = KMeans(n_clusters=2, n_init=5, random_state=0).fit(lab[idx])
    lbl = km.predict(lab)
    bright = int(np.argmax(km.cluster_centers_[:, 0]))
    return (lbl == bright).reshape(bgr.shape[:2])


def postprocess(mask, m_per_px, close_px=7, open_px=5, min_area_m2=150, exclude=None):
    """Closing (tutup celah) -> opening (buang bintik) -> buang objek < min_area_m2.
    exclude = mask boolean area yang dikecualikan (mis. daratan), opsional."""
    m = mask.astype(np.uint8) * 255
    if exclude is not None:
        m[exclude] = 0
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (close_px, close_px)))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_px, open_px)))
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m, 8)
    min_px = min_area_m2 / (m_per_px ** 2)
    keep = np.zeros(n, bool)
    keep[1:] = st[1:, cv2.CC_STAT_AREA] >= min_px
    return keep[lbl]


def polygon_mask(shape, polygons):
    """Mask dari daftar poligon [[(x,y),...], ...], untuk masker daratan manual."""
    m = np.zeros(shape[:2], np.uint8)
    for poly in polygons:
        cv2.fillPoly(m, [np.array(poly, np.int32)], 1)
    return m.astype(bool)


# ------------------------------------------------------------------ 4. luas & estimasi
def area_and_units(mask, m_per_px, unit_area_m2=49.0):
    area_m2 = float(mask.sum()) * m_per_px ** 2
    return {"luas_m2": round(area_m2, 1), "luas_ha": round(area_m2 / 1e4, 3),
            "luas_per_unit_m2": round(unit_area_m2, 2),
            "estimasi_unit": int(round(area_m2 / unit_area_m2))}


# ------------------------------------------------------------------ 5. kunci jawaban & evaluasi
def read_answer_key(path, shape=None):
    """Baca kunci jawaban PNG: piksel merah (R tinggi, G & B rendah, tidak transparan) = KJA."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGRA)
    if img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
    b, g, r, a = [img[:, :, i].astype(int) for i in range(4)]
    m = (r > 150) & (g < 100) & (b < 100) & (a > 127)
    if shape is not None and m.shape != tuple(shape[:2]):
        raise ValueError(f"Ukuran kunci jawaban {m.shape} beda dengan citra {tuple(shape[:2])}")
    return m


def evaluate(pred, gt):
    tp = int((pred & gt).sum()); fp = int((pred & ~gt).sum()); fn = int((~pred & gt).sum())
    tn = int((~pred & ~gt).sum())
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    acc = (tp + tn) / pred.size
    return {"IoU": round(iou, 4), "precision": round(prec, 4), "recall": round(rec, 4),
            "F1": round(f1, 4), "akurasi_piksel": round(acc, 4)}


def error_map(rgb, pred, gt):
    """Hijau = benar (TP), merah = salah tandai (FP), biru = terlewat (FN)."""
    o = rgb.astype(np.float32).copy()
    for m, col in [(pred & gt, (40, 200, 90)), (pred & ~gt, (230, 40, 40)), (~pred & gt, (40, 120, 255))]:
        o[m] = o[m] * 0.35 + np.array(col) * 0.65
    return o.astype(np.uint8)


def answer_key_rows(gt):
    """Pisahkan kunci jawaban jadi baris-baris keramba (komponen terhubung) untuk kalibrasi."""
    n, lbl, st, cent = cv2.connectedComponentsWithStats(gt.astype(np.uint8), 8)
    rows = [{"id": i, "luas_px": int(st[i, cv2.CC_STAT_AREA]), "pusat": (int(cent[i][0]), int(cent[i][1]))}
            for i in range(1, n) if st[i, cv2.CC_STAT_AREA] > 30]
    return lbl, rows


def calibrate_unit_area(gt_lbl, counts, m_per_px):
    """counts = {id_komponen: jumlah_kotak}. Hasil: rata-rata luas satu kotak (m²)."""
    per = []
    for cid, n in counts.items():
        px = int((gt_lbl == cid).sum())
        if n > 0 and px > 0:
            per.append(px * m_per_px ** 2 / n)
    if not per:
        raise ValueError("counts kosong atau id tidak ditemukan")
    return float(np.mean(per)), float(np.std(per)), per


def overlay(rgb, mask, color=(255, 122, 40), alpha=0.5):
    o = rgb.astype(np.float32).copy()
    o[mask] = o[mask] * (1 - alpha) + np.array(color) * alpha
    return o.astype(np.uint8)

In [ ]:
import os, glob, shutil
import numpy as np, cv2, pandas as pd
import matplotlib.pyplot as plt
import importlib, kja_pipeline as kp
importlib.reload(kp)

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.rcParams['figure.dpi'] = 110
def tampil(gambar, judul, ukuran=(14, 4.5), cmap=None):
    fig, ax = plt.subplots(1, len(gambar), figsize=ukuran)
    ax = np.atleast_1d(ax)
    for a, im, t in zip(ax, gambar, judul):
        a.imshow(im, cmap=cmap); a.set_title(t, loc='left', fontsize=10); a.axis('off')
    plt.tight_layout(); plt.show()
print('Siap. Berjalan di Colab:', IN_COLAB)

## 1. Upload citra dan kunci jawaban
Upload **potongan citra** (misalnya `jatiluhur_padat_potongan.png`) dan, kalau sudah ada, **kunci jawabannya** (misalnya `jatiluhur_padat_kunci.png` dari Dayan). Boleh upload beberapa file sekaligus.

In [ ]:
if IN_COLAB:
    terunggah = files.upload()
    print('File terunggah:', list(terunggah))
print('File gambar di folder kerja:', sorted(glob.glob('*.png') + glob.glob('*.jpg')))

## 2. Pengaturan
Isi nama file dan skala citra. Skala (meter per piksel) diukur dari garis skala Google Earth:

| Citra | `M_PER_PX` | `CROP_BAWAH` |
|---|---|---|
| `jatiluhur_padat_potongan.png` (Dayan) | 0.891 | 0 |
| `jatiluhur_tanaman_padat_potongan.png` (Aufa) | 0.873 | 0 |
| Citra utuh hasil Save Image (ada logo dan garis skala di bawah) | sesuai garis skala | 40 |

`KUNCI_JAWABAN` dikosongkan (`''`) kalau belum ada. `JUMLAH_KOTAK` diisi di bagian 7.

In [ ]:
CITRA          = 'jatiluhur_padat_potongan.png'
M_PER_PX       = 0.891
CROP_BAWAH     = 0
KUNCI_JAWABAN  = 'jatiluhur_padat_kunci.png'   # '' kalau belum ada

# Parameter metode (nilai awal hasil uji coba; bisa diuji ulang di bagian 8)
WIN        = 15     # ukuran jendela GLCM (piksel), sekitar 13 m
STEP       = 4      # jarak geser jendela GLCM (piksel); lebih besar = lebih cepat, lebih kasar
K          = 3      # jumlah klaster K-Means: air, tanaman, KJA
FRAC       = 0.4    # klaster dianggap KJA bila kerapatan garisnya >= FRAC x klaster tertinggi
GATE_T     = 0.06   # syarat minimal kerapatan garis rangka di sekitar piksel
REFINE_PX  = 13     # ukuran closing saat memangkas tepi ke rangka; 0 = tidak dipangkas
MIN_AREA   = 150    # objek lebih kecil dari ini (m²) dibuang
LUAS_UNIT  = 64.1   # luas satu unit KJA (m²); hasil kalibrasi 10 baris (Dayan+Aufa), lihat bagian 7. Standar Taskov dkk.: 49.0

## 3. Preprocessing

In [ ]:
bgr = kp.load_image(CITRA, CROP_BAWAH)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
enh = kp.preprocess(bgr)
h, w = enh.shape
print(f'Ukuran citra: {w} x {h} piksel  ≈  {w*M_PER_PX:.0f} m x {h*M_PER_PX:.0f} m  ({w*h*M_PER_PX**2/1e4:.2f} ha)')
tampil([rgb, gray, enh], ['Citra asli', 'Grayscale', 'Median filter + CLAHE'], cmap='gray')

## 4. Ekstraksi fitur
**Tekstur GLCM** mengukur pola hubungan antar-piksel bertetangga di dalam jendela kecil. Area keramba berpola grid, sehingga contrast dan entropy tinggi; air polos, sehingga homogeneity dan energy tinggi.

Proses ini paling lama (sekitar 1–3 menit tergantung ukuran citra).

In [ ]:
glcm = kp.glcm_maps(enh, win=WIN, step=STEP)
tampil([glcm[p] for p in kp.GLCM_PROPS], [f'GLCM {p}' for p in kp.GLCM_PROPS], ukuran=(18, 3.8), cmap='magma')

**Warna hijau relatif** membedakan tanaman mengapung (hijau) dari rangka keramba. Nilainya dikurangi warna latar sekitar supaya beda warna antar-citra satelit (mosaik) tidak ikut terbaca.

**Kerapatan garis rangka**: rangka keramba berupa garis terang yang tipis. Garis seperti ini diambil dengan operasi morfologi *white top-hat*, lalu dihitung kerapatannya per jendela. Tanaman dan hutan bertekstur, tetapi tidak punya garis tipis yang terang, sehingga fitur ini memisahkan keduanya dari keramba.

In [ ]:
hijau = kp.green_relative(bgr, WIN)
garis = kp.line_density(bgr, enh, WIN)
tampil([hijau, garis], ['Warna hijau relatif', 'Kerapatan garis rangka'], ukuran=(14, 4), cmap='magma')

## 5. Segmentasi K-Means

In [ ]:
fitur = {**glcm, 'hijau': hijau, 'line_density': garis}
label, _, ringkasan = kp.segment_kmeans(fitur, k=K, kja_by='line_density')
klaster_kja = kp.select_kja_clusters(ringkasan, 'line_density', FRAC)
display(pd.DataFrame(ringkasan).set_index('klaster'))
print('Klaster yang dianggap KJA:', klaster_kja)

warna_klaster = np.array([[60, 140, 255], [255, 122, 40], [70, 200, 90], [200, 80, 200], [240, 220, 60]])[:K]
tampil([(rgb * .5 + warna_klaster[label] * .5).astype(np.uint8), kp.overlay(rgb, np.isin(label, klaster_kja))],
       ['Hasil K-Means (tiap warna = satu klaster)', 'Klaster KJA sebelum post-processing'])

## 6. Post-processing dan hasil
1. **Syarat kerapatan garis**: buang area bertekstur yang tidak punya garis rangka (hamparan tanaman, hutan).
2. **Pangkas tepi ke rangka**: batas KJA disesuaikan dengan garis rangka, supaya tidak melebar ke air.
3. **Morfologi** (closing, opening) dan **filter luas**.

In [ ]:
kandidat = np.isin(label, klaster_kja) & kp.line_gate(garis, threshold=GATE_T)
kasar = kp.postprocess(kandidat, M_PER_PX, min_area_m2=MIN_AREA)
hasil = kp.postprocess(kp.refine_to_frame(kasar, enh, REFINE_PX), M_PER_PX, close_px=3, open_px=3,
                       min_area_m2=MIN_AREA) if REFINE_PX else kasar
warna_saja = kp.color_baseline(bgr)

tampil([kp.overlay(rgb, warna_saja, alpha=.8), kp.overlay(rgb, kasar), kp.overlay(rgb, hasil)],
       [f'Pembanding: warna saja ({warna_saja.mean()*100:.1f}%)',
        f'Tekstur + garis, sebelum pangkas tepi ({kasar.mean()*100:.1f}%)',
        f'Hasil akhir ({hasil.mean()*100:.1f}%)'], ukuran=(18, 4.5))

est = kp.area_and_units(hasil, M_PER_PX, LUAS_UNIT)
print(f"Luas area KJA       : {est['luas_m2']:,.0f} m²  ({est['luas_ha']} ha)")
print(f"Luas per unit       : {est['luas_per_unit_m2']} m²")
print(f"Estimasi jumlah unit: {est['estimasi_unit']:,} unit")

## 7. Evaluasi terhadap kunci jawaban dan kalibrasi luas per unit
Bagian ini berjalan kalau `KUNCI_JAWABAN` sudah diisi dan filenya sudah di-upload.

**Peta kesalahan:** hijau = benar, merah = salah tandai (bukan KJA tapi dianggap KJA), biru = terlewat.

In [ ]:
ADA_KUNCI = bool(KUNCI_JAWABAN) and os.path.exists(KUNCI_JAWABAN)
if ADA_KUNCI:
    kunci = kp.read_answer_key(KUNCI_JAWABAN, bgr.shape)
    tabel = pd.DataFrame({
        'Warna saja': kp.evaluate(warna_saja, kunci),
        'Tekstur + garis': kp.evaluate(kasar, kunci),
        'Hasil akhir (+ pangkas tepi)': kp.evaluate(hasil, kunci),
    }).T
    display(tabel)
    tampil([kp.overlay(rgb, kunci, (230, 30, 30), .5), kp.error_map(rgb, hasil, kunci)],
           [f'Kunci jawaban ({kunci.mean()*100:.1f}% area)', 'Peta kesalahan hasil akhir'], ukuran=(15, 4.5))
else:
    print('Kunci jawaban belum ada. Bagian evaluasi dilewati.')

**Kalibrasi luas per unit.** Gambar di bawah menampilkan setiap baris keramba di kunci jawaban beserta nomor ID-nya. Cocokkan dengan 5 baris yang dihitung Dayan atau Aufa, lalu isi `JUMLAH_KOTAK` dengan format `{ID: jumlah_kotak}`, contoh `{12: 16, 25: 8}`. Jalankan ulang sel setelah diisi.

In [ ]:
JUMLAH_KOTAK = {}   # contoh: {12: 16, 25: 8, 31: 10, 40: 12, 52: 14}

if ADA_KUNCI:
    kunci_lbl, baris = kp.answer_key_rows(kunci)
    fig, ax = plt.subplots(figsize=(15, 7.5))
    ax.imshow(kp.overlay(rgb, kunci, (230, 30, 30), .45)); ax.axis('off')
    for b in baris:
        ax.text(*b['pusat'], str(b['id']), color='white', fontsize=8, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='black', alpha=.6, lw=0))
    ax.set_title('ID tiap baris keramba di kunci jawaban', loc='left'); plt.show()
    if JUMLAH_KOTAK:
        rata, sd, per = kp.calibrate_unit_area(kunci_lbl, JUMLAH_KOTAK, M_PER_PX)
        print('Luas per kotak tiap baris (m²):', [round(p, 1) for p in per])
        print(f'Rata-rata luas satu kotak: {rata:.1f} ± {sd:.1f} m²  (standar Taskov dkk.: 49 m²)')
        LUAS_UNIT = rata
        est = kp.area_and_units(hasil, M_PER_PX, LUAS_UNIT)
        est_kunci = kp.area_and_units(kunci, M_PER_PX, LUAS_UNIT)
        print(f"Estimasi jumlah unit dari hasil program : {est['estimasi_unit']:,}")
        print(f"Estimasi jumlah unit dari kunci jawaban: {est_kunci['estimasi_unit']:,}")
        selisih = (est['estimasi_unit'] - est_kunci['estimasi_unit']) / max(1, est_kunci['estimasi_unit']) * 100
        print(f'Selisih: {selisih:+.1f}%')
    else:
        print('Isi JUMLAH_KOTAK untuk menghitung luas per unit. Sementara dipakai 49 m².')

## 8. Uji parameter (opsional)
Mencari kombinasi ukuran jendela GLCM dan syarat kerapatan garis dengan IoU tertinggi. Butuh kunci jawaban. Ubah `JALANKAN_UJI = True` untuk menjalankan; prosesnya beberapa menit karena GLCM dihitung ulang untuk tiap ukuran jendela.

In [ ]:
JALANKAN_UJI = False
DAFTAR_WIN    = [11, 15, 21]
DAFTAR_GATE   = [0.03, 0.06, 0.10]
DAFTAR_REFINE = [0, 9, 13, 17]

if JALANKAN_UJI and ADA_KUNCI:
    baris_uji = []
    for wn in DAFTAR_WIN:
        print(f'Jendela {wn} px ...')
        g = kp.glcm_maps(enh, win=wn, step=STEP, verbose=False)
        hj, gr = kp.green_relative(bgr, wn), kp.line_density(bgr, enh, wn)
        lb, _, rk = kp.segment_kmeans({**g, 'hijau': hj, 'line_density': gr}, k=K)
        kl = kp.select_kja_clusters(rk, 'line_density', FRAC)
        for gt in DAFTAR_GATE:
            ks = kp.postprocess(np.isin(lb, kl) & kp.line_gate(gr, threshold=gt), M_PER_PX, min_area_m2=MIN_AREA)
            for rp in DAFTAR_REFINE:
                hs = kp.postprocess(kp.refine_to_frame(ks, enh, rp), M_PER_PX, 3, 3, MIN_AREA) if rp else ks
                baris_uji.append({'WIN': wn, 'GATE_T': gt, 'REFINE_PX': rp, **kp.evaluate(hs, kunci)})
    hasil_uji = pd.DataFrame(baris_uji).sort_values('IoU', ascending=False)
    display(hasil_uji.head(10))
    hasil_uji.to_csv('uji_parameter.csv', index=False)
else:
    print('Uji parameter tidak dijalankan.')

## 9. Simpan hasil
Semua hasil disimpan ke folder `hasil/` lalu di-download sebagai satu file zip.

In [ ]:
nama = os.path.splitext(os.path.basename(CITRA))[0]
os.makedirs('hasil', exist_ok=True)
cv2.imwrite(f'hasil/{nama}_mask.png', hasil.astype(np.uint8) * 255)
cv2.imwrite(f'hasil/{nama}_overlay.png', cv2.cvtColor(kp.overlay(rgb, hasil), cv2.COLOR_RGB2BGR))
ringkas = {'citra': CITRA, 'm_per_px': M_PER_PX, 'WIN': WIN, 'GATE_T': GATE_T, 'REFINE_PX': REFINE_PX, **est}
if ADA_KUNCI:
    ringkas.update({f'eval_{k}': v for k, v in kp.evaluate(hasil, kunci).items()})
    cv2.imwrite(f'hasil/{nama}_peta_kesalahan.png', cv2.cvtColor(kp.error_map(rgb, hasil, kunci), cv2.COLOR_RGB2BGR))
pd.DataFrame([ringkas]).to_csv(f'hasil/{nama}_ringkasan.csv', index=False)
display(pd.DataFrame([ringkas]).T.rename(columns={0: 'nilai'}))
shutil.make_archive(f'hasil_{nama}', 'zip', 'hasil')
if IN_COLAB:
    files.download(f'hasil_{nama}.zip')
print('Tersimpan:', sorted(os.listdir('hasil')))

## 10. Estimasi total se-Waduk Jatiluhur (gabungan banyak zona)
Dipakai setelah screenshot dari Dayan dan Aufa masuk (5 zona, beberapa zona butuh 2 potongan). Upload semua file di bagian 1 dulu, lalu isi daftar di bawah: satu baris per potongan citra (satu zona bisa punya 2 baris kalau dicover 2 screenshot).

`m_per_px` diisi dari garis skala masing-masing citra (bisa beda tiap screenshot). `crop_bawah` diisi 40 kalau citranya hasil Save Image utuh (ada logo/garis skala di bawah), 0 kalau sudah dipotong bersih.

Kalau 2 potongan di zona yang sama saling tumpang tindih dikit di tengah, areanya mungkin sedikit kehitung dobel — wajar untuk estimasi, catat di laporan sebagai batasan. Kalau tumpang tindihnya banyak, potong dulu salah satu citra sebelum upload.

In [ ]:
DAFTAR_ZONA = [
    # {'zona': 'Padat',         'file': 'zona_padat_a.jpg',   'm_per_px': 0.89, 'crop_bawah': 40},
    # {'zona': 'Padat',         'file': 'zona_padat_b.jpg',   'm_per_px': 0.89, 'crop_bawah': 40},
    # {'zona': 'Tanaman Padat', 'file': 'zona4_tanaman.jpg',  'm_per_px': 0.87, 'crop_bawah': 40},
    # {'zona': 'Zona 1',        'file': 'zona1_a.jpg',        'm_per_px': 0.89, 'crop_bawah': 40},
    # {'zona': 'Zona 1',        'file': 'zona1_b.jpg',        'm_per_px': 0.89, 'crop_bawah': 40},
]

hasil_zona = []
for item in DAFTAR_ZONA:
    bgr_z = kp.load_image(item['file'], item.get('crop_bawah', 40))
    enh_z = kp.preprocess(bgr_z)
    glcm_z = kp.glcm_maps(enh_z, win=WIN, step=STEP, verbose=False)
    hijau_z = kp.green_relative(bgr_z, WIN)
    garis_z = kp.line_density(bgr_z, enh_z, WIN)
    label_z, _, ringkasan_z = kp.segment_kmeans({**glcm_z, 'hijau': hijau_z, 'line_density': garis_z}, k=K, kja_by='line_density')
    klaster_z = kp.select_kja_clusters(ringkasan_z, 'line_density', FRAC)
    kandidat_z = np.isin(label_z, klaster_z) & kp.line_gate(garis_z, threshold=GATE_T)
    kasar_z = kp.postprocess(kandidat_z, item['m_per_px'], min_area_m2=MIN_AREA)
    hasil_z = kp.postprocess(kp.refine_to_frame(kasar_z, enh_z, REFINE_PX), item['m_per_px'], 3, 3, MIN_AREA) if REFINE_PX else kasar_z
    est_z = kp.area_and_units(hasil_z, item['m_per_px'], LUAS_UNIT)
    hasil_zona.append({'zona': item['zona'], 'file': item['file'], **est_z})
    print(f"{item['zona']:16s} {item['file']:22s} luas={est_z['luas_m2']:>9,.0f} m2   unit={est_z['estimasi_unit']:>5,}")

if hasil_zona:
    tabel_zona = pd.DataFrame(hasil_zona)
    display(tabel_zona)
    per_zona = tabel_zona.groupby('zona', sort=False).agg(luas_m2=('luas_m2', 'sum'), estimasi_unit=('estimasi_unit', 'sum')).reset_index()
    display(per_zona)
    print()
    print(f"TOTAL se-Waduk Jatiluhur: {per_zona['luas_m2'].sum():,.0f} m2 ({per_zona['luas_m2'].sum()/1e4:.1f} ha), estimasi {per_zona['estimasi_unit'].sum():,} unit")
    per_zona.to_csv('hasil/ringkasan_total_waduk.csv', index=False)
else:
    print('DAFTAR_ZONA masih kosong. Isi dulu dengan file-file yang sudah di-upload.')